In [1]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="../data/sample_texts.txt",
    column_names=["text", "label"]
)

print(dataset["train"][0])

{'text': 'I love deep learning', 'label': 1}


In [2]:
def clean_labels(example):
    example["label"] = int(example["label"])
    return example

dataset = dataset.map(clean_labels)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 7
    })
})

In [3]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding=True,
        truncation=True
    )

dataset = dataset.map(tokenize, batched=True)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 7
    })
})

In [4]:
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch")

In [5]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    evaluation_strategy="no"   # important si dataset petit
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"]
)

c:\Users\adilu\anaconda3\envs\tp4_clean\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [7]:
trainer.train()

  0%|          | 0/5 [00:00<?, ?it/s]

c:\Users\adilu\anaconda3\envs\tp4_clean\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'train_runtime': 3.8642, 'train_samples_per_second': 9.058, 'train_steps_per_second': 1.294, 'train_loss': 0.6520804405212403, 'epoch': 5.0}


TrainOutput(global_step=5, training_loss=0.6520804405212403, metrics={'train_runtime': 3.8642, 'train_samples_per_second': 9.058, 'train_steps_per_second': 1.294, 'total_flos': 63387720060.0, 'train_loss': 0.6520804405212403, 'epoch': 5.0})

In [8]:
text = "I love NLP"

inputs = tokenizer(text, return_tensors="pt", truncation=True)

outputs = model(**inputs)
print(outputs.logits)

tensor([[-0.2899, -0.0934]], grad_fn=<AddmmBackward0>)


In [9]:
import torch

labels = ["negative", "positive"]

pred = torch.argmax(outputs.logits, dim=1)
print("Prediction:", labels[pred])

Prediction: positive
